# Clean Partition 8 by Dropping Missing Rows

Chunked cleaner for Jane Street partition 8.

What this does:
- Reads partition 8 in batches.
- Drops rows missing `responder_6`, `weight`, or any `feature_` value.
- Keeps `date_id`, `time_id`, `symbol_id`, `weight`, `responder_6`, and all `feature_` columns.
- Treats `feature_09`, `feature_10`, `feature_11`, `symbol_id`, and `time_id` as discrete columns to preserve.
- Writes cleaned parquet incrementally.

It does not train a model and does not load the full partition into pandas.

## 1. Setup

In [ ]:
from pathlib import Path
import gc
import shutil

import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"

def resolve_partition_path(partition_id, data_dir=DATA_DIR):
    candidates = [
        data_dir / "train.parquet" / f"partition_id={partition_id}",
        data_dir / "train.parquet" / f"partition_id={partition_id}.parquet",
        data_dir / f"partition_id={partition_id}",
        data_dir / f"partition_id={partition_id}.parquet",
        data_dir / f"part_{partition_id}.parquet",
    ]
    existing = [path for path in candidates if path.exists()]
    if not existing:
        checked = "\n".join(str(path) for path in candidates)
        raise FileNotFoundError(f"Could not find partition {partition_id}. Checked:\n{checked}")
    return existing[0]

PARTITION8_PATH = resolve_partition_path(8)
OUTPUT_DIR = DATA_DIR / "clean" / "partition_8_drop_missing"

print(f"partition 8 path: {PARTITION8_PATH}")
print(f"output dir: {OUTPUT_DIR}")

## 2. Schema and columns

In [ ]:
TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
ID_COLS = ["date_id", "time_id", "symbol_id"]
DISCRETE_COLS = ["symbol_id", "time_id", "feature_09", "feature_10", "feature_11"]

dataset = ds.dataset(PARTITION8_PATH, format="parquet")
columns = dataset.schema.names
FEATURE_COLS = [col for col in columns if col.startswith("feature_")]
PRESENT_ID_COLS = [col for col in ID_COLS if col in columns]
PRESENT_DISCRETE_COLS = [col for col in DISCRETE_COLS if col in columns]
SELECTED_COLS = PRESENT_ID_COLS + [WEIGHT_COL] + FEATURE_COLS + [TARGET_COL]
DROP_MISSING_COLS = FEATURE_COLS + [TARGET_COL, WEIGHT_COL]

missing_required = [col for col in [TARGET_COL, WEIGHT_COL] if col not in columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")
if not FEATURE_COLS:
    raise ValueError("No feature_ columns found")

print(f"feature count: {len(FEATURE_COLS)}")
print(f"ID columns: {PRESENT_ID_COLS}")
print(f"discrete columns to preserve: {PRESENT_DISCRETE_COLS}")
print(f"selected column count: {len(SELECTED_COLS)}")

## 3. Small dry run

In [ ]:
BATCH_SIZE = 8_192

scanner = dataset.scanner(
    columns=SELECTED_COLS,
    batch_size=BATCH_SIZE,
    batch_readahead=1,
    fragment_readahead=1,
    use_threads=False,
)

dry_batch = next(scanner.to_batches()).to_pandas(split_blocks=True, self_destruct=True)
dry_clean = dry_batch.dropna(subset=DROP_MISSING_COLS)

print(f"dry batch rows before drop: {len(dry_batch):,}")
print(f"dry batch rows after drop:  {len(dry_clean):,}")
print(f"dry batch dropped rows:     {len(dry_batch) - len(dry_clean):,}")
print(f"dry clean missing values:   {int(dry_clean[DROP_MISSING_COLS].isna().sum().sum())}")
display(dry_clean[PRESENT_ID_COLS + PRESENT_DISCRETE_COLS + [WEIGHT_COL, TARGET_COL]].head())

del dry_batch, dry_clean, scanner
gc.collect()

## 4. Chunked clean writer

In [ ]:
OVERWRITE_OUTPUT = False

def cast_clean_batch(batch_df):
    """Keep types compact and preserve discrete columns as integer-coded columns."""
    for col in FEATURE_COLS:
        if col in PRESENT_DISCRETE_COLS:
            continue
        batch_df[col] = batch_df[col].astype(np.float32)

    batch_df[WEIGHT_COL] = batch_df[WEIGHT_COL].astype(np.float32)
    batch_df[TARGET_COL] = batch_df[TARGET_COL].astype(np.float32)

    for col in PRESENT_DISCRETE_COLS:
        if col in batch_df.columns:
            batch_df[col] = batch_df[col].astype("int16")

    if "date_id" in batch_df.columns:
        batch_df["date_id"] = batch_df["date_id"].astype("int16")

    return batch_df

def clean_partition_drop_missing(parquet_path, output_dir, batch_size=BATCH_SIZE, overwrite=False):
    output_dir = Path(output_dir)
    if output_dir.exists():
        if not overwrite:
            raise FileExistsError(f"Output already exists: {output_dir}")
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    dataset = ds.dataset(parquet_path, format="parquet")
    scanner = dataset.scanner(
        columns=SELECTED_COLS,
        batch_size=batch_size,
        batch_readahead=1,
        fragment_readahead=1,
        use_threads=False,
    )

    writer = None
    rows_read = 0
    rows_written = 0
    rows_dropped = 0
    output_file = output_dir / "part-0.parquet"

    try:
        for batch_idx, record_batch in enumerate(scanner.to_batches(), start=1):
            batch_df = record_batch.to_pandas(split_blocks=True, self_destruct=True)
            rows_read += len(batch_df)

            clean_df = batch_df.dropna(subset=DROP_MISSING_COLS).copy()
            rows_dropped += len(batch_df) - len(clean_df)

            if clean_df.empty:
                del batch_df, clean_df
                gc.collect()
                continue

            clean_df = cast_clean_batch(clean_df)
            table = pa.Table.from_pandas(clean_df[SELECTED_COLS], preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(output_file, table.schema, compression="zstd")
            writer.write_table(table)
            rows_written += len(clean_df)

            if batch_idx % 25 == 0:
                print(
                    f"batch {batch_idx:,} | read {rows_read:,} | "
                    f"written {rows_written:,} | dropped {rows_dropped:,}"
                )

            del batch_df, clean_df, table
            gc.collect()
    finally:
        if writer is not None:
            writer.close()

    summary = {
        "output_file": str(output_file),
        "rows_read": rows_read,
        "rows_written": rows_written,
        "rows_dropped": rows_dropped,
        "drop_rate": rows_dropped / rows_read if rows_read else np.nan,
    }
    return summary

clean_summary = clean_partition_drop_missing(
    PARTITION8_PATH,
    OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    overwrite=OVERWRITE_OUTPUT,
)
display(pd.Series(clean_summary).to_frame("value"))

## 5. Verify cleaned output

In [ ]:
if OUTPUT_DIR.exists():
    clean_dataset = ds.dataset(OUTPUT_DIR, format="parquet")
    print(clean_dataset.schema)

    verify_scanner = clean_dataset.scanner(
        columns=DROP_MISSING_COLS,
        batch_size=BATCH_SIZE,
        batch_readahead=1,
        fragment_readahead=1,
        use_threads=False,
    )
    verify_rows = 0
    verify_missing = 0
    for record_batch in verify_scanner.to_batches():
        verify_df = record_batch.to_pandas(split_blocks=True, self_destruct=True)
        verify_rows += len(verify_df)
        verify_missing += int(verify_df.isna().sum().sum())
        del verify_df
        gc.collect()

    print(f"verified rows: {verify_rows:,}")
    print(f"verified missing values in target/weight/features: {verify_missing:,}")
